# 🗃️ Step 2: Database Building

In this notebook, we take the **cleaned dataset** (`products_clean.csv`) from Step 1 and load it into a structured **SQLite database** (`chatbot.db`).

The database will contain 4 tables:
1. `products` — All cleaned product data (our main table)
2. `conversations` — For storing chat memory (used by P5)
3. `viewed_products` — Tracking which products the user viewed (used by P5)
4. `admin_logs` — Logging admin actions like add/edit/delete (used by P5)

In [ ]:
import pandas as pd

# Load the cleaned dataset from Step 1
df = pd.read_csv("../data/products_clean.csv")

print("✅ Clean dataset loaded!")
print(f"📊 Shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

✅ Clean dataset loaded!
📊 Shape: (913, 25)
📋 Columns: ['product_id', 'title', 'product_description', 'rating', 'ratings_count', 'initial_price', 'discount', 'final_price', 'currency', 'images', 'delivery_options', 'product_details', 'breadcrumbs', 'product_specifications', 'amount_of_stars', 'what_customers_said', 'seller_name', 'sizes', 'videos', 'seller_information', 'variations', 'best_offer', 'more_offers', 'category', 'is_active']


### Create the SQLite Database & Tables
- We create 4 tables: `products`, `conversations`, `viewed_products`, `admin_logs`.
- We also add performance indexes on frequently queried columns.

In [ ]:
import sqlite3
import os

# Set database path
db_path = "../database/chatbot.db"

# Create the database folder if it doesn't exist
os.makedirs(os.path.dirname(db_path), exist_ok=True)

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row  # So we can access columns by name

# ════════════ Table 1: products ════════════
conn.execute("""
    CREATE TABLE IF NOT EXISTS products (
        product_id TEXT PRIMARY KEY,
        title TEXT NOT NULL,
        product_description TEXT,
        rating REAL DEFAULT 0,
        ratings_count INTEGER DEFAULT 0,
        initial_price REAL,
        discount REAL DEFAULT 0,
        final_price REAL NOT NULL,
        currency TEXT DEFAULT 'USD',
        category TEXT,
        breadcrumbs TEXT,
        product_specifications TEXT,
        what_customers_said TEXT,
        images TEXT,
        seller_name TEXT,
        variations TEXT,
        is_active INTEGER DEFAULT 1,
        created_at DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

# ═══ Table 2: conversations (P5 will use this for memory) ═══
conn.execute("""
    CREATE TABLE IF NOT EXISTS conversations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id TEXT NOT NULL,
        role TEXT NOT NULL,
        content TEXT NOT NULL,
        intent TEXT,
        sentiment REAL,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

# ═══ Table 3: viewed_products (P5 will track what user saw) ═══
conn.execute("""
    CREATE TABLE IF NOT EXISTS viewed_products (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id TEXT NOT NULL,
        product_id TEXT NOT NULL,
        product_title TEXT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

# ═══ Table 4: admin_logs (P5 will log admin actions) ═══
conn.execute("""
    CREATE TABLE IF NOT EXISTS admin_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        action TEXT NOT NULL,
        product_id TEXT NOT NULL,
        details TEXT,
        admin_user TEXT DEFAULT 'admin',
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

conn.commit()
print("✅ All 4 tables created successfully!")

✅ All 4 tables created successfully!


### Create Performance Indexes
- Indexes speed up frequent queries like filtering by category, price, or rating.

In [3]:
# Indexes for faster product queries
conn.execute("CREATE INDEX IF NOT EXISTS idx_products_category ON products(category)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_products_price ON products(final_price)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_products_rating ON products(rating)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_products_active ON products(is_active)")

# Index for faster conversation lookups
conn.execute("CREATE INDEX IF NOT EXISTS idx_conversations_session ON conversations(session_id)")

conn.commit()
print("✅ Indexes created for: category, price, rating, is_active, session_id")

✅ Indexes created for: category, price, rating, is_active, session_id


### Insert Products into the Database
- We loop through the cleaned DataFrame and insert each product into the `products` table.
- Using `INSERT OR REPLACE` so we can re-run this safely without duplicates.

In [4]:
inserted = 0

for _, row in df.iterrows():
    conn.execute("""
        INSERT OR REPLACE INTO products 
        (product_id, title, product_description, rating, ratings_count,
         initial_price, discount, final_price, currency, category,
         breadcrumbs, product_specifications, what_customers_said,
         images, seller_name, variations, is_active)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        str(row.get('product_id', '')),
        row['title'],
        row['product_description'],
        row['rating'],
        row['ratings_count'],
        row['initial_price'],
        row['discount'],
        row['final_price'],
        row['currency'],
        row['category'],
        row.get('breadcrumbs', ''),
        row['product_specifications'],
        row['what_customers_said'],
        row.get('images', ''),
        row.get('seller_name', 'Unknown Seller'),
        row.get('variations', 'Standard'),
        row.get('is_active', 1)
    ))
    inserted += 1

conn.commit()
print(f"✅ {inserted} products inserted into the database!")

✅ 913 products inserted into the database!


> ### Notes:
 - We convert `product_id` to string since the schema defines it as `TEXT PRIMARY KEY`.
 - `INSERT OR REPLACE` ensures that if we re-run the notebook, it won't crash on duplicates.

---

### Verification: Check the Database
- Let's verify that everything was inserted correctly by running a few queries.

In [5]:
# 1. Total products count
total = conn.execute("SELECT COUNT(*) FROM products WHERE is_active = 1").fetchone()[0]
print(f"📦 Total Active Products: {total}")

# 2. Average price
avg_price = conn.execute("SELECT AVG(final_price) FROM products WHERE is_active = 1").fetchone()[0]
print(f"💰 Average Price: ${round(avg_price, 2)}")

# 3. Average rating
avg_rating = conn.execute("SELECT AVG(rating) FROM products WHERE is_active = 1 AND rating > 0").fetchone()[0]
print(f"⭐ Average Rating: {round(avg_rating, 2)}")

# 4. Number of categories
cat_count = conn.execute("SELECT COUNT(DISTINCT category) FROM products WHERE is_active = 1").fetchone()[0]
print(f"📂 Total Categories: {cat_count}")

# 5. Top 5 categories by count
print("\n📊 Top 5 Categories:")
rows = conn.execute("""
    SELECT category, COUNT(*) as cnt 
    FROM products WHERE is_active = 1 
    GROUP BY category ORDER BY cnt DESC LIMIT 5
""").fetchall()
for r in rows:
    print(f"   - {r[0]}: {r[1]} products")

📦 Total Active Products: 913
💰 Average Price: $20.44
⭐ Average Rating: 4.08
📂 Total Categories: 95

📊 Top 5 Categories:
   - tops: 109 products
   - dresses: 96 products
   - shirts: 84 products
   - jeans: 47 products
   - sports-shoes: 45 products


In [6]:
# Quick check: fetch one product and display its data
sample = conn.execute("SELECT * FROM products LIMIT 1").fetchone()
print("--- Sample Product ---")
for key in sample.keys():
    print(f"  {key}: {sample[key]}")

--- Sample Product ---
  product_id: 8376765
  title: Lino Perros
  product_description: Women Navy Blue Solid Backpack
  rating: 3.8
  ratings_count: 15
  initial_price: 47.94
  discount: 58.0
  final_price: 47.94
  currency: USD
  category: backpacks
  breadcrumbs: [{"name":"Accessories","url":"https://www.myntra.com/accessories"},{"name":"Women","url":"https://www.myntra.com/women-accessories"},{"name":"Backpacks","url":"https://www.myntra.com/backpacks"},{"name":"Lino Perros","url":"https://www.myntra.com/lino-perros-backpacks"},{"name":"More by Lino Perros","url":"https://www.myntra.com/lino-perros"}]
  product_specifications: "specification_name":"Add-Ons","specification_value":"NA","specification_name":"Back","specification_value":"Non-Padded","specification_name":"Compartment Closure","specification_value":"Flap","specification_name":"External Pocket","specification_value":"Zip Pocket","specification_name":"Features","specification_value":"NA","specification_name":"Features 2",

### Verify Table Structure
- Let's also confirm all 4 tables exist and list their schemas.

In [7]:
# List all tables in the database
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("📋 Tables in chatbot.db:")
for t in tables:
    print(f"   ✅ {t[0]}")

# Close connection
conn.close()
print(f"\n✅ Database saved successfully to: {db_path}")
print(f"📁 File size: {os.path.getsize(db_path) / 1024:.1f} KB")

📋 Tables in chatbot.db:
   ✅ products
   ✅ conversations
   ✅ sqlite_sequence
   ✅ viewed_products
   ✅ admin_logs

✅ Database saved successfully to: ../database/chatbot.db
📁 File size: 3576.0 KB
